In [1]:
import torch 
import numpy as np 
import pufferlib 
from pufferlib.ocean.breakout.breakout import Breakout
import torch.nn as nn
import pufferlib.vector
from pufferlib.ocean.breakout.breakout import Breakout
import time

/root/backtoRL/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
class breakout_ppo: 
  def __init__(self, n_envs): 
    self.env = pufferlib.vector.make(
      Breakout,
      env_kwargs={"num_envs": n_envs},
      num_envs = 1
    )
    self.n_envs = n_envs 
    

In [3]:
import inspect
import pufferlib.pufferl
print(inspect.getsource(pufferlib.pufferl))

# puffer [train | eval | sweep] [env_name] [optional args] -- See https://puffer.ai for full details
# This is the same as python -m pufferlib.pufferl [train | eval | sweep] [env_name] [optional args]
# Distributed example: torchrun --standalone --nnodes=1 --nproc-per-node=6 -m pufferlib.pufferl train puffer_nmmo3

import warnings
warnings.filterwarnings('error', category=RuntimeWarning)

import os
import sys
import glob
import ast
import time
import random
import shutil
import argparse
import importlib
import configparser
from threading import Thread
from collections import defaultdict, deque

import numpy as np
import psutil

import torch
import torch.distributed
from torch.distributed.elastic.multiprocessing.errors import record
import torch.utils.cpp_extension

import pufferlib
import pufferlib.sweep
import pufferlib.vector
import pufferlib.pytorch
try:
    from pufferlib import _C
except ImportError:
    raise ImportError('Failed to import C/CUDA advantage kernel. If you have non-

In [4]:


# ── Policy ──────────────────────────────────────────────────────────────
class Policy(nn.Module):
    def __init__(self, obs_dim=118, n_actions=3):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(obs_dim, 128), nn.ReLU(),
            nn.Linear(128, 128),     nn.ReLU(),
        )
        self.actor  = nn.Linear(128, n_actions)
        self.critic = nn.Linear(128, 1)

    def forward(self, x):
        x = self.trunk(x.float())
        return self.actor(x), self.critic(x).squeeze(-1)

# ── GAE ──────────────────────────────────────────────────────────────────
def compute_gae(rewards, values, dones, gamma, gae_lambda):
    # [horizon, n_envs]
    advantages = torch.zeros_like(rewards)
    last_adv = 0
    for t in reversed(range(rewards.shape[0])):
        next_val = values[t+1] if t+1 < rewards.shape[0] else 0
        delta    = rewards[t] + gamma * next_val * (1 - dones[t]) - values[t]
        last_adv = delta + gamma * gae_lambda * (1 - dones[t]) * last_adv
        advantages[t] = last_adv
    return advantages

# ── Hyperparams ──────────────────────────────────────────────────────────
device      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_envs      = 1024*4
horizon     = 128
batch_size  = n_envs * horizon
n_epochs    = 4
minibatch   = 512
clip_coef   = 0.2
vf_coef     = 0.5
ent_coef    = 0.01
gamma       = 0.99
gae_lambda  = 0.95
lr          = 2.5e-4
total_steps = 1_000_000

# ── Setup ────────────────────────────────────────────────────────────────
env = pufferlib.vector.make(
    Breakout,
    env_kwargs={"num_envs": 1024},  # as in breakout.ini
    backend=pufferlib.vector.Multiprocessing,  # implicit default in CLI
    num_envs=8,  # as in [vec] num_envs
    num_workers='auto',
    batch_size='auto',
    zero_copy=True,
)
policy = Policy().to(device)
optimizer = torch.optim.Adam(policy.parameters(), lr=lr)

# Rollout buffers [horizon, n_envs, ...] — all on GPU
obs_buf  = torch.zeros(horizon, n_envs, 118, device=device)
act_buf  = torch.zeros(horizon, n_envs,      device=device, dtype=torch.long)
logp_buf = torch.zeros(horizon, n_envs,      device=device)
val_buf  = torch.zeros(horizon, n_envs,      device=device)
rew_buf  = torch.zeros(horizon, n_envs,      device=device)
done_buf = torch.zeros(horizon, n_envs,      device=device)

# ── Training loop ────────────────────────────────────────────────────────
obs, _ = env.reset()
global_step = 0
t_start = time.perf_counter()

while global_step < total_steps:

    # ── Collect rollout ──────────────────────────────────────────────────
    for t in range(horizon):
        with torch.no_grad():
            o = torch.tensor(obs, dtype=torch.float32, device=device)
            logits, value = policy(o)
            dist   = torch.distributions.Categorical(logits=logits)
            action = dist.sample()
            logp   = dist.log_prob(action)

        next_obs, rewards, dones, truncs, infos = env.step(action.cpu().numpy())

        obs_buf[t]  = o
        act_buf[t]  = action
        logp_buf[t] = logp
        val_buf[t]  = value
        rew_buf[t]  = torch.tensor(rewards, device=device)
        done_buf[t] = torch.tensor(dones,   device=device)

        obs = next_obs
        global_step += n_envs

    # ── Compute advantages ───────────────────────────────────────────────
    advantages = compute_gae(rew_buf, val_buf, done_buf, gamma, gae_lambda)
    returns    = advantages + val_buf

    # ── PPO update ───────────────────────────────────────────────────────
    obs_flat  = obs_buf.reshape(-1, 118)
    act_flat  = act_buf.reshape(-1)
    logp_flat = logp_buf.reshape(-1)
    val_flat  = val_buf.reshape(-1)
    adv_flat  = advantages.reshape(-1)
    ret_flat  = returns.reshape(-1)
    adv_flat  = (adv_flat - adv_flat.mean()) / (adv_flat.std() + 1e-8)

    for _ in range(n_epochs):
        idx = torch.randperm(batch_size, device=device)
        for start in range(0, batch_size, minibatch):
            mb = idx[start:start+minibatch]
            logits, value = policy(obs_flat[mb])
            dist    = torch.distributions.Categorical(logits=logits)
            newlogp = dist.log_prob(act_flat[mb])
            entropy = dist.entropy().mean()

            ratio   = (newlogp - logp_flat[mb]).exp()
            adv     = adv_flat[mb]
            pg_loss = -torch.min(
                ratio * adv,
                torch.clamp(ratio, 1-clip_coef, 1+clip_coef) * adv
            ).mean()

            vf_loss = 0.5 * (value - ret_flat[mb]).pow(2).mean()
            loss    = pg_loss + vf_coef*vf_loss - ent_coef*entropy

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(policy.parameters(), 0.5)
            optimizer.step()

    if global_step % 50_000 < n_envs * horizon:
        print(f"step={global_step:>7} | pg={pg_loss.item():.3f} | vf={vf_loss.item():.3f} | ent={entropy.item():.3f}")


t_end = time.perf_counter()
print(f"Time taken: {t_end - t_start:.2f} seconds") 
steps_per_second = total_steps / (t_end - t_start)
print(f"Steps per second: {steps_per_second:.2f}")


step= 524288 | pg=0.103 | vf=0.005 | ent=1.094
step=1048576 | pg=-0.011 | vf=0.019 | ent=1.081
Time taken: 84.38 seconds
Steps per second: 11850.94


In [5]:
env = pufferlib.vector.make(Breakout, num_envs=1, backend=pufferlib.vector.Serial)
obs, _ = env.reset()

total_reward = 0
done = False

while not done:
    with torch.no_grad():
        o = torch.tensor(obs, dtype=torch.float32).to(device)
        logits, _ = policy(o)
        dist = torch.distributions.Categorical(logits=logits)
        action = dist.sample().cpu().numpy()

    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward[0]
    done = terminated[0] or truncated[0]

print(f"Total reward: {total_reward}")
env.close()

Total reward: 1.0


In [6]:
import sys
sys.argv = ['puffer', 'train', 'puffer_breakout']

from pufferlib.pufferl import train
train('puffer_breakout')

Usage: puffer [--load-model-path LOAD_MODEL_PATH] [--load-id LOAD_ID]
              [--render-mode {auto,human,ansi,rgb_array,raylib,None}]
              [--save-frames SAVE_FRAMES] [--gif-path GIF_PATH] [--fps FPS]
              [--max-runs MAX_RUNS] [--wandb] [--wandb-project WANDB_PROJECT]
              [--wandb-group WANDB_GROUP] [--neptune]
              [--neptune-name NEPTUNE_NAME]
              [--neptune-project NEPTUNE_PROJECT] [--local-rank LOCAL_RANK]
              [--tag TAG] [--package PACKAGE] [--env-name ENV_NAME]
              [--policy-name POLICY_NAME] [--rnn-name RNN_NAME]
              [--max-suggestion-cost MAX_SUGGESTION_COST]
              [--vec.backend VEC.BACKEND] [--vec.num-envs VEC.NUM_ENVS]
              [--vec.num-workers VEC.NUM_WORKERS]
              [--vec.batch-size VEC.BATCH_SIZE]
              [--vec.zero-copy VEC.ZERO_COPY] [--vec.seed VEC.SEED]
              [--env.num-envs ENV.NUM_ENVS] [--env.frameskip ENV.FRAMESKIP]
              [--policy.hidd

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ /root/backtoRL/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3747 in       │
│ run_code                                                                                         │
│                                                                                                  │
│   3744 │   │   │   │   if async_:                                                                │
│   3745 │   │   │   │   │   await eval(code_obj, self.user_global_ns, self.user_ns)               │
│   3746 │   │   │   │   else:                                                                     │
│ ❱ 3747 │   │   │   │   │   exec(code_obj, self.user_global_ns, self.user_ns)                     │
│   3748 │   │   │   finally:                                                                      │
│   3749 │   │   │   │   # Reset our crash handler in place                                        │
│   3750 │   │   │   │   sys.excepthook = old_excepthook                                           │
│                                                                                                  │
│ in <module>:5                                                                                    │
│                                                                                                  │
│   2 sys.argv = ['puffer', 'train', 'puffer_breakout']                                            │
│   3                                                                                              │
│   4 from pufferlib.pufferl import train                                                          │
│ ❱ 5 train('puffer_breakout')                                                                     │
│   6                                                                                              │
│                                                                                                  │
│ /root/backtoRL/.venv/lib/python3.12/site-packages/pufferlib/pufferl.py:865 in train              │
│                                                                                                  │
│    862 │   │   return f'{data_dir}/{model_file}'                                                 │
│    863                                                                                           │
│    864 def train(env_name, args=None, vecenv=None, policy=None, logger=None):                    │
│ ❱  865 │   args = args or load_config(env_name)                                                  │
│    866 │   vecenv = vecenv or load_env(env_name, args)                                           │
│    867 │   policy = policy or load_policy(args, vecenv)                                          │
│    868                                                                                           │
│                                                                                                  │
│ /root/backtoRL/.venv/lib/python3.12/site-packages/pufferlib/pufferl.py:1169 in load_config       │
│                                                                                                  │
│   1166 │   │   action='help', help='Show this help message and exit')                            │
│   1167 │                                                                                         │
│   1168 │   # Unpack to nested dict                                                               │
│ ❱ 1169 │   parsed = vars(parser.parse_args())                                                    │
│   1170 │   args = defaultdict(dict)                                                              │
│   1171 │   for key, value in parsed.items():                                                     │
│   1172 │   │   next = args                                                                       │
│                                                            

/root/backtoRL/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3755: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
print(inspect.getsource(pufferlib.pufferl.PuffeRL.train))

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱ 1 print(inspect.getsource(pufferlib.pufferl.PuffeRL.train))                                    │
│   2                                                                                              │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
NameError: name 'inspect' is not defined

In [ ]:
#so breakout is being trained at like fucking 100ksps by puffer, and out naive one is at like 10-20k sps SADGE 